# SD1.5 Out-of-Range Weighted Recovery-CFG Ablation

This study fixes each Christoffel sampling distribution at its CFG-7.5 estimate and varies only the recovery conditioning. The CFG 1 control reuses all five compatible main-study trials; CFG 3, 5, and 7.5 use five new trials. All runs use the weighted unitary Fourier operator, $\zeta=1/2$, and the 2,000-step main weighted learning-rate schedule.


In [ ]:
from pathlib import Path
import importlib.util
import os

cwd = Path.cwd().resolve()
candidates = [cwd / 'analyze_results/weighted/ablation']
candidates.extend(parent / 'analyze_results/weighted/ablation' for parent in (cwd, *cwd.parents))
STUDY_ROOT = next((path for path in candidates if (path / 'analysis.py').is_file()), None)
if STUDY_ROOT is None:
    raise FileNotFoundError('Could not locate analyze_results/weighted/ablation/analysis.py')
spec = importlib.util.spec_from_file_location('weighted_cfg_ablation_analysis', STUDY_ROOT / 'analysis.py')
diagnostic = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(diagnostic)

SCENARIO = 'out_of_range'
OUTPUT_DIR = diagnostic.RESULT_ROOT / SCENARIO / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
diagnostic.PROJECT_ROOT, STUDY_ROOT, OUTPUT_DIR


## Optimization Trajectories and Trial Uncertainty

Each figure fixes one sampling distribution and separates the five sampling ratios into subplots. Colors denote recovery CFG, faint curves are individual trials, bold curves are arithmetic means, and shading is one sample standard deviation.


In [ ]:
TRACES = diagnostic.load_optimization_traces(SCENARIO)
TRACE_SUMMARY = diagnostic.optimization_trace_summary(TRACES)
TRACE_OUTPUTS = diagnostic.export_optimization_trace_figures(
    TRACES, OUTPUT_DIR, show_uncertainty=False, show=True
)
print(f'Loaded {len(TRACES):,} saved optimization points')
TRACE_OUTPUTS


## Load Results

LPIPS is stored during reconstruction and filled incrementally when needed. The completion table distinguishes reused main-study controls from new CFG-ablation lines.


In [ ]:
LPIPS_TABLE = diagnostic.ensure_lpips(SCENARIO, device='cpu')
ROWS = diagnostic.load_rows(SCENARIO)
COMPLETION = diagnostic.completion_table(ROWS)
print(f'Loaded {len(ROWS)} / {diagnostic.EXPECTED_ROWS_PER_SCENARIO} analysis rows')
print(f'New ablation target: {diagnostic.EXPECTED_NEW_ROWS_PER_SCENARIO} rows')
display(diagnostic.count_table(ROWS))
display(COMPLETION.groupby(['sampling_law', 'recovery_line'], as_index=False)[['observed', 'expected', 'left']].sum())
display(ROWS[['distribution_key', 'line_condition', 'source_kind', 'samp_perc', 'repeat_id', 'psnr_db', 'ssim', 'lpips', 'bp_best_loss']].head())


## Metric Curves

Following the main notebooks, PSNR, SSIM, LPIPS, and best weighted loss are rendered as separate TeX-styled sampling-ratio figures. Solid lines show trial means and shading shows one sample standard deviation.


In [ ]:
METRIC_OUTPUTS = diagnostic.plot_metric_curves(ROWS, output_dir=OUTPUT_DIR, show=True)
METRIC_OUTPUTS


## Reconstruction Panel

Each row is one fixed CFG-7.5 sampling law. Within each recovery column, the displayed trial minimizes LPIPS, with PSNR used to break ties. Change `PANEL_RATIO` to any value in the 1--5% grid.


In [ ]:
PANEL_RATIO = 0.01
PANEL_OUTPUTS = diagnostic.plot_reconstruction_panel(
    ROWS,
    sampling_ratio=PANEL_RATIO,
    output_dir=OUTPUT_DIR,
    show=True,
)
PANEL_OUTPUTS


## Aggregate Metric Forest Plot

Horizontal intervals are 95% confidence intervals; stars mark the best observed sampling-law and recovery-CFG pair for each metric.


In [ ]:
ROWS = diagnostic.load_rows(SCENARIO)
FOREST_OUTPUT = diagnostic.plot_aggregate_forest(
    ROWS,
    output_dir=OUTPUT_DIR,
    show=True,
)
FOREST_OUTPUT
